# 10. IONIQ 5 검색 품질 확인 — M6 기준선과 M7 개선 실험

**질문 → 검색된 상위 5개 → 미리 정한 정답 근거와 비교**하는 노트북입니다.

- 목표: **Hit@5 ≥ 0.7 / MRR ≥ 0.5**
- 대상: `NE1_2027_ko_KR.pdf`, 기존 청크 2,338개, `text-embedding-3-small` 1,536차원
- 평가셋: PDF 근거를 확인해 만든 **24개 개발용 예비 질문**. AI 작성 라벨이며 팀 검토 전입니다.
- 정답 페이지와 근거 구절을 먼저 정하고 청크 ID로 연결했습니다. 그 후 검색을 실행했으며 결과를 보고 정답 라벨을 바꾸지 않았습니다.
- 질문은 정답이 있는 단일 질문입니다. 되묻기, 대화 맥락 처리, 최종 답변의 정확성·안전성 평가는 별도입니다.

기본 실행은 저장된 결과를 읽기만 합니다. 재측정 셀의 스위치를 켤 때만 OpenAI API를 호출합니다. 설명서 DB와 기존 임베딩·배포 앱은 수정하지 않습니다.


## 1. 점수의 뜻

| 지표 | 쉬운 뜻 | 계산 |
|---|---|---|
| Hit@5 | 검색 결과 5개 안에 정답 근거가 하나라도 들어왔나? | 성공 질문 수 ÷ 전체 질문 수 |
| MRR | 정답 근거가 얼마나 앞에 나왔나? | 질문마다 `1 / 첫 정답 순위`를 계산한 뒤 평균 |
| MRR@5 | 상위 5개만 보는 보수적인 MRR | 5위 밖은 0점 |

정답이 1위면 1점, 2위면 0.5점, 5위면 0.2점입니다. 여기서는 전체 2,338개 순위 기준 MRR과 MRR@5를 모두 기록했습니다. 수업에서 MRR의 검색 범위를 정했다면 그 기준에 맞춰 제출하세요.

**Hit@5가 70%라는 것은 답변 정확도가 70%라는 뜻이 아닙니다.** 검색 근거를 찾는 능력입니다. 부모 본문을 불러온 후의 보완 효과도 이 점수에는 포함하지 않습니다.


In [1]:
# 노트북을 어느 폴더에서 열어도 저장소 루트를 찾습니다.
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next(p for p in candidates if (p / 'src/car_search_rag/anna_rag/evaluation').is_dir())
eval_dir = project_root / 'src/car_search_rag/anna_rag/evaluation'
if str(eval_dir) not in sys.path:
    sys.path.insert(0, str(eval_dir))

def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))

dataset = read_json(eval_dir / 'pilot_v1.json')
print('예비 질문 수:', len(dataset['questions']))
print('라벨 상태: AI 작성 / 팀 검토 필요')


예비 질문 수: 24
라벨 상태: AI 작성 / 팀 검토 필요


## 2. 미리 정한 정답표

페이지가 같다는 이유만으로 정답으로 세지 않습니다. 근거 구절이 실제로 들어 있는 청크 ID를 정답으로 등록했습니다. 이미지 설명에도 답의 근거가 들어 있으면 포함합니다. 검색된 본문과 이미지 설명 청크 모두 5개 자리를 차지합니다.

원문과 같은 의미의 다른 청크가 정답 목록에서 빠져 있을 수 있습니다. 팀원이 검토해 수정할 경우 `pilot_v2.json`처럼 새 버전으로 저장하고 **M6와 M7 둘 다 재평가**하세요. 한쪽 점수만 유리하게 바꾸지 않습니다.


In [2]:
# 전체 질문과 PDF 기준 페이지, 근거 구절을 함께 확인합니다.
question_rows = [{
    'ID': q['id'], '분야': q['category'], '질문': q['question'],
    'PDF 페이지': q['evidence_pdf_pages'], '근거 구절': q['evidence_excerpt'],
    '정답 청크 수': len(q['relevant_chunk_ids']),
} for q in dataset['questions']]
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display(pd.DataFrame(question_rows))


,ID,분야,질문,PDF 페이지,근거 구절,정답 청크 수
0,I5-01,충전,충전이 끝났는데 차량에서 케이블이 안 빠져. 비상 해제 케이블은 어디에 있어?,[18],화물칸을 열고 비상 해제,2
1,I5-02,충전,예약 충전을 해놨는데 지금 바로 충전하고 싶으면 어떻게 해?,[24],차량을 즉시 충전하려면 충전 도어를 열고,2
2,I5-03,충전,충전 중인데 바로 멈추려면 차에서 어떤 버튼을 눌러야 해?,[24],15초 이내,1
3,I5-04,충전,배터리를 80퍼센트까지만 충전하게 설정할 수 있어?,[30],50-100,1
4,I5-05,충전,충전 목표를 현재 잔량보다 낮게 해놓으면 어떻게 돼?,[31],현재 배터리 충전량보다 낮게,2
5,I5-06,V2L,캠핑에서 차 전기를 쓸 때 배터리가 너무 줄지 않게 제한하려면?,[31],20 % 이상 80 % 이하,2
6,I5-07,V2L,차 안에서 노트북 충전할 콘센트는 어디에 있어?,[38],전원 패널은 뒷좌석 아래,2
7,I5-08,비상,차가 고장 나 견인해야 하는데 바퀴를 땅에 대고 끌어도 돼?,[45],모든 바퀴를 들어올려,1
8,I5-09,키,스마트키 건전지를 사려는데 규격이 뭐야?,[159],CR2450,1
9,I5-10,키,차 밖에서 스마트키로 원격 시동을 걸려면 어떤 순서로 눌러?,[159],4초 이내에 스마트 키의 원격 시동,1


## 3. 계산 검증

아래 예시는 실제 모델을 부르지 않고 계산식만 확인합니다. 여러 정답이 있어도 가장 먼저 등장한 정답 하나의 순위를 사용합니다.


In [3]:
from evaluate_retrieval import measure
assert measure(['a', 'b', 'c'], ['b'])['rr'] == 0.5
assert measure(['a', 'b', 'c', 'd', 'e', 'f'], ['f'])['hit'] == 0
assert measure(['a', 'b', 'c', 'd', 'e', 'f'], ['f'], k=6)['rr'] == 1 / 6
assert measure(['a', 'b'], ['a', 'b'])['rank'] == 1
print('순위·상위 5개 밖 실패·복수 정답 계산 검증 통과')


순위·상위 5개 밖 실패·복수 정답 계산 검증 통과


## 4. 실제 측정 결과

M6는 배포 코드의 검색 함수와 동일한 모델·필터·상위 5개 설정으로 **원래 질문**을 검색한 기준선입니다. M7은 질문을 LLM으로 간결한 검색 문장으로 바꾼 뒤 같은 검색기를 사용합니다.

MRR은 전체 순위 기준이며 MRR@5도 나란히 표시합니다. 목표 통과 여부는 이 예비셋에서만 유효합니다.


In [4]:
m6_dir = eval_dir / 'm6_pilot_v1'
m7_dir = eval_dir / 'm7_rewrite_pilot_v1/measurement'
m6_summary = read_json(m6_dir / 'summary.json')
m7_summary = read_json(m7_dir / 'summary.json')
m6 = read_json(m6_dir / 'results.json')
m7 = read_json(m7_dir / 'results.json')
# 같은 문서 스냅샷과 같은 원래 질문·정답으로 비교했는지 검증합니다.
assert m6_summary['corpus_sha256'] == m7_summary['corpus_sha256']
assert [(r['id'], r['question'], r['relevant_chunk_ids']) for r in m6] == [(r['id'], r['question'], r['relevant_chunk_ids']) for r in m7]
comparison = pd.DataFrame([
    {'실험': name, '질문 수': s['question_count'], 'Hit@5': s['hit_at_5'],
     'MRR': s['mrr_full'], 'MRR@5': s['mrr_at_5'],
     '두 목표 통과': s['hit_at_5'] >= .7 and s['mrr_full'] >= .5}
    for name, s in [('M6 원래 질문', m6_summary), ('M7 질문 재작성', m7_summary)]
])
display(comparison)


,실험,질문 수,Hit@5,MRR,MRR@5,두 목표 통과
0,M6 원래 질문,24,0.625000,0.457080,0.430556,False
1,M7 질문 재작성,24,0.708333,0.521162,0.492361,True


## 이번 실행의 결론

| 지표 | M6 기준선 | M7 질문 재작성 | 참고 목표 |
|---|---:|---:|---:|
| Hit@5 | 0.6250 (15/24) | 0.7083 (17/24) | ≥ 0.7 |
| MRR — 전체 순위 | 0.4571 | 0.5212 | ≥ 0.5 |
| MRR@5 | 0.4306 | 0.4924 | MRR을 @5로 정의한다면 ≥ 0.5 |

M7에서 새로 성공한 질문은 I5-03, 06, 07, 09이고, 기존 성공에서 실패로 바뀐 질문은 I5-10, 23입니다. 남아 있는 실패는 I5-10, 13, 15, 17, 20, 21, 23입니다.

**전체 순위 MRR 기준으로는 예비 목표를 넘었지만, MRR@5 기준으로는 아직 미달입니다.** 수업의 MRR 계산 범위를 먼저 확인해야 합니다. 질문 하나가 바뀌면 Hit@5가 약 4.17%p 바뀌는 작은 표본입니다.

실패 원인 기록:
- I5-07: 일상 질문에서 콘센트 위치가 밀렸으나, “전원 소켓(AC 콘센트)”으로 재작성한 뒤 573위 → 1위. 용어 차이라는 가설을 지지합니다.
- I5-09: “건전지”를 “배터리 규격”으로 다듬어 67위 → 1위.
- I5-13: “트렁크 높이 저장”이 여전히 원하는 테일게이트 근거를 찾지 못함(15위 → 81위). 재작성만으로 해결되지 않았습니다.
- I5-10: 원래 있던 “어떤 순서로 눌러”가 일반적인 “원격 시동 방법”으로 축약돼 1위 → 6위. 질문의 구체성을 잃을 수 있습니다.
- I5-23: “몇 개월마다”가 “교체 주기”로 바뀌며 2위 → 7위. 이 변화의 인과관계는 추가 검증이 필요합니다.

**권장 결정: 아직 운영에 일괄 적용하지 않고, 팀의 정답 라벨 검토 및 별도 검증 질문으로 확인합니다.**


## 5. 질문별로 좋아졌는지 확인

전체 평균만 보면 일부 질문이 나빠진 것을 놓칠 수 있습니다. 각 질문의 성공 여부와 정답 순위를 함께 봅니다.


In [5]:
rows = []
for before, after in zip(m6, m7):
    rows.append({'ID': before['id'], '질문': before['question'],
                 'M6 정답 순위': before['first_relevant_rank_full'],
                 'M7 정답 순위': after['first_relevant_rank_full'],
                 'M6 Hit@5': before['hit_at_5'], 'M7 Hit@5': after['hit_at_5'],
                 'M7 검색 질문': after.get('search_question', after['question'])})
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display(pd.DataFrame(rows))


,ID,질문,M6 정답 순위,M7 정답 순위,M6 Hit@5,M7 Hit@5,M7 검색 질문
0,I5-01,충전이 끝났는데 차량에서 케이블이 안 빠져. 비상 해제 케이블은 어디에 있어?,2,3,1,1,충전 완료 후 충전 케이블이 분리되지 않을 때 비상 충전 커넥터 해제 케이블 위치는?
1,I5-02,예약 충전을 해놨는데 지금 바로 충전하고 싶으면 어떻게 해?,2,2,1,1,예약 충전 중 즉시 충전으로 전환하는 방법?
2,I5-03,충전 중인데 바로 멈추려면 차에서 어떤 버튼을 눌러야 해?,6,1,0,1,충전 중 충전을 즉시 중지하려면 차량에서 어떤 버튼을 눌러야 하나요?
3,I5-04,배터리를 80퍼센트까지만 충전하게 설정할 수 있어?,1,1,1,1,배터리 충전 상한을 80%로 설정할 수 있나요?
4,I5-05,충전 목표를 현재 잔량보다 낮게 해놓으면 어떻게 돼?,2,1,1,1,충전 목표 설정값을 현재 배터리 잔량보다 낮게 설정하면 어떻게 되나요?
5,I5-06,캠핑에서 차 전기를 쓸 때 배터리가 너무 줄지 않게 제한하려면?,12,5,0,1,캠핑 중 차량 배터리 방전을 방지하도록 전원 사용을 제한하는 방법?
6,I5-07,차 안에서 노트북 충전할 콘센트는 어디에 있어?,573,1,0,1,차량 내 전원 소켓(AC 콘센트)은 어디에 있나요?
7,I5-08,차가 고장 나 견인해야 하는데 바퀴를 땅에 대고 끌어도 돼?,2,5,1,1,고장 차량을 바퀴 접지 상태로 견인해도 되나요?
8,I5-09,스마트키 건전지를 사려는데 규격이 뭐야?,67,1,0,1,스마트키(리모컨 키) 배터리 규격은 무엇인가?
9,I5-10,차 밖에서 스마트키로 원격 시동을 걸려면 어떤 순서로 눌러?,1,6,1,0,스마트키로 차량 외부에서 원격 시동을 거는 방법?


## 6. 실패 원문을 직접 살펴보기

`selected_id`를 바꾸면 질문별 Top-5 전체 내용을 볼 수 있습니다. 순위가 낮은 이유를 단어 불일치, 비슷한 기능 혼동, 본문·이미지 설명 중복, 정답 라벨 누락 등으로 구분해 검토하세요.


In [6]:
selected_id = 'I5-07'
for name, results in [('M6', m6), ('M7', m7)]:
    selected = next(r for r in results if r['id'] == selected_id)
    print(name, selected['question'])
    print('정답 PDF 페이지:', selected['evidence_pdf_pages'])
    print('확인한 근거:', selected['evidence_excerpt'])
    print('전체 정답 순위:', selected['first_relevant_rank_full'])
    for doc in selected['top5']:
        print('검색 순위:', doc['rank'], '| 종류:', doc['chunk_type'], '| PDF:', doc['pdf_pages'])
        print('정답 라벨에 포함:', doc['chunk_id'] in selected['relevant_chunk_ids'])
        print(doc['page_content'])


M6 차 안에서 노트북 충전할 콘센트는 어디에 있어?
정답 PDF 페이지: [38]
확인한 근거: 전원 패널은 뒷좌석 아래
전체 정답 순위: 573
검색 순위: 1 | 종류: image | PDF: [189]
정답 라벨에 포함: False
목차: 편의 장치 > 운전석 자세 메모리 시스템 사용하기 > 승하차 편의 기능
이미지 설명: 승하차 편의 기능의 그림. 주변 본문(자동 연결, 이미지 해석 아님):
승하차 편의 기능
운전자가 편하게 차량에 탑승하거나 하차할 수 있도록 좌석 위치를 자동으로 조절합니다.
인포테인먼트 시스템에서 차량 > 시트 > 승하차 편의 > 운전석 시트 이동을 누른 후 운전석의 작동 방법을 설
정할 수 있습니다( 길게 이동/ 짧게 이동/ 끄기).
•	시트 위치 변경 알림을 선택하면 시트 위치 변경 시 변경 정보가 화면에 표시됩니다.
검색 순위: 2 | 종류: text | PDF: [189]
정답 라벨에 포함: False
목차: 편의 장치 > 운전석 자세 메모리 시스템 사용하기 > 승하차 편의 기능
본문:
승하차 편의 기능

운전자가 편하게 차량에 탑승하거나 하차할 수 있도록 좌석 위치를 자동으로 조절합니다.
인포테인먼트 시스템에서 차량 > 시트 > 승하차 편의 > 운전석 시트 이동을 누른 후 운전석의 작동 방법을 설
정할 수 있습니다( 길게 이동/ 짧게 이동/ 끄기).
•	시트 위치 변경 알림을 선택하면 시트 위치 변경 시 변경 정보가 화면에 표시됩니다.
[알아두기]
업데이트로 인해 인포테인먼트 시스템의 상세 설정이 변경될 수 있습니다. 자세한 설정 방법은 인포테인먼트 시
스템 웹 매뉴얼을 참고하십시오.

하차 위치로 운전석 이동
'P'(주차)로 변속한 후 시동을 끄고 운전석 도어를 열면 운전석이 설정 메뉴에서 선택한 거리만큼 뒤로 이동합니
다.
검색 순위: 3 | 종류: text | PDF: [51]
정답 라벨에 포함: False
목차: 차량 살펴보기 > 차량 내부 > 대시보드 및 중앙 콘솔
본문:
11.	USB 단자 ............

## 7. 원인 분석과 M7 기록

- M6에서 “노트북 충전할 콘센트”는 “차량 내부 전원 패널” 근거가 573위, “스마트키 건전지 규격”은 배터리 교체 근거가 67위, “쉬라고 알림”은 운전자 주의 경고 근거가 314위였습니다.
- 일상 표현과 설명서 명칭의 차이, 질문의 부가 표현이 검색을 흐리는 현상을 **가설**로 삼았습니다. 이것만으로 임베딩 모델 자체의 결함이 입증된 것은 아닙니다.
- M7은 모든 24개 질문에 동일한 재작성 프롬프트를 적용합니다. 재작성 모델은 `gpt-5.6-luna`이며 모델에 정답·PDF 근거·기존 검색 결과를 보내지 않았습니다. 실제 입력 질문과 재작성문, 프롬프트는 JSON으로 보관했습니다.
- 재작성은 의도를 바꿀 위험이 있습니다. 예를 들어 I5-20에서는 원래 질문에 없던 “자전거”가 추가됐습니다. 좋은 점수만 보고 바로 배포하지 않습니다.
- 동일 예비셋을 원인 분석과 개선 비교에 사용했으므로 독립적인 테스트 결과가 아닙니다. 운영 효과를 주장하려면 별도 질문셋으로 다시 검증해야 합니다.
- 점수는 **고정된 정답 라벨 기준**입니다. 팀 검토 후 의미가 같은 추가 근거를 인정하면 점수가 바뀔 수 있습니다.

지금 배포된 챗봇에는 M7 실험을 적용하지 않았습니다. 평가 결과를 읽는 것과 운영 검색 방식을 바꾸는 것은 별도 작업입니다.


## 8. 다시 측정하기 — 선택

평소에는 `RUN_EVALUATION = False`로 둡니다. True이면 질문 임베딩 API를 호출합니다. M7까지 실행하면 질문 재작성 LLM 호출도 추가됩니다. 기존 문서 임베딩은 재생성하지 않습니다.

결과는 시각이 붙은 새 폴더에 저장하며 기존 M6/M7 결과를 덮어쓰지 않습니다. 로컬 `.env`가 저장소 루트에 있어야 합니다. 이 노트북의 비교 셀은 최초 기록 폴더를 읽으므로 새 결과와 비교하려면 4번 셀의 경로를 변경하세요.


In [7]:
RUN_EVALUATION = False
RUN_M7 = False
if RUN_EVALUATION:
    from datetime import datetime
    from evaluate_retrieval import evaluate
    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    fresh_summary = evaluate(eval_dir / 'pilot_v1.json', eval_dir / f'm6_{stamp}')
    display(fresh_summary)
    if RUN_M7:
        from experiment_query_rewrite import run
        display(run(eval_dir / f'm7_{stamp}'))
else:
    print('API 호출 없이 저장된 평가 결과만 확인했습니다.')


API 호출 없이 저장된 평가 결과만 확인했습니다.


## 9. 팀에 공유할 때

1. **M6 기준선**: 질문 수, 문서·모델, Hit@5·MRR·MRR@5와 계산 범위를 같이 표시합니다.
2. **실패 사례**: 질문, 정답 근거, 실제 검색 결과, 원인 가설을 함께 제시합니다.
3. **M7 실험**: 바꾼 한 가지 요소와 같은 질문 기준 전후 수치, 악화된 사례도 기록합니다.
4. **한계**: 24개 AI 작성 개발 예비셋, 라벨 팀 검토 전, 독립 검증 전, 답변 정확도와 다른 지표임을 밝힙니다.

제출 전에는 팀원이 PDF 원문과 정답 청크를 검토하고, 개발에 쓰지 않은 질문으로 다시 측정하는 것이 좋습니다.
